# 84. The Multi-Facility Location: p-Center Problem

## Tier 3: The Advanced Algorithm (Tabu Search)

### Key assumptions
- Maintain memory of recent moves to avoid cycling and escape local optima
- Tabu list prevents revisiting recently examined solutions
- Aspiration criteria allow overriding tabu restrictions for exceptional improvements
- Adaptive neighborhood structure considers facility relocations and demand reassignments

### Approach (step-by-step)
Tabu Search provides sophisticated exploration with memory-based guidance:

1. **Initialization**: Start with an initial feasible solution
2. **Tabu List Management**: Maintain memory of recent facility moves
3. **Neighborhood Exploration**: Generate diverse candidate solutions
4. **Aspiration Criteria**: Override tabu restrictions for exceptional moves
5. **Intensification/Diversification**: Balance between exploitation and exploration
6. **Termination**: Stop when convergence criteria are met

### What to look for in the results
- Convergence behavior showing escape from local optima
- Tabu list effectiveness in preventing cycling
- Solution quality improvement over basic heuristics
- Robustness across different problem instances
- Computational efficiency vs solution quality trade-offs

### Concrete example (from the source)
We'll implement Tabu Search with the following specifications:

**Expected Output:**
```
Initial solution: {1, 3, 5}
Initial objective: 1.581
Iteration 12: New best = 1.414
Iteration 28: New best = 1.118
Iteration 45: New best = 0.707
Final best solution: {0, 2, 4}
Final best objective: 0.707
```

In [1]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple, Set, Dict, Optional
import random
import time
from collections import deque
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

In [ ]:
@dataclass
class PCenterInstance:
    """Data class for p-center problem instances"""
    demand_points: List[Tuple[float, float]]
    facility_locations: List[Tuple[float, float]]
    p: int
    
    def compute_distance_matrix(self) -> np.ndarray:
        """Compute Euclidean distance matrix"""
        n_demand = len(self.demand_points)
        n_facilities = len(self.facility_locations)
        distances = np.zeros((n_demand, n_facilities))
        
        for i, (dx, dy) in enumerate(self.demand_points):
            for j, (fx, fy) in enumerate(self.facility_locations):
                distances[i, j] = np.sqrt((dx - fx)**2 + (dy - fy)**2)
        
        return distances
    
    def evaluate_solution(self, facility_set: Set[int]) -> float:
        """Evaluate maximum distance for a given set of facilities"""
        distances = self.compute_distance_matrix()
        max_distance = 0.0
        
        for i in range(len(self.demand_points)):
            min_dist = min(distances[i, j] for j in facility_set)
            max_distance = max(max_distance, min_dist)
        
        return max_distance
    
    def get_facility_assignments(self, facility_set: Set[int]) -> Dict[int, int]:
        """Get which facility serves each demand point"""
        distances = self.compute_distance_matrix()
        assignments = {}
        
        for i in range(len(self.demand_points)):
            min_dist = float('inf')
            best_facility = None
            
            for j in facility_set:
                if distances[i, j] < min_dist:
                    min_dist = distances[i, j]
                    best_facility = j
            
            assignments[i] = best_facility
        
        return assignments

In [ ]:
class TabuSearchPCenter:
    """Tabu Search algorithm for p-center problem"""
    
    def __init__(self, instance: PCenterInstance, tabu_tenure: int = 5, 
                 max_iterations: int = 100, aspiration_threshold: float = 0.1):
        self.instance = instance
        self.distances = instance.compute_distance_matrix()
        self.n_demand = len(instance.demand_points)
        self.n_facilities = len(instance.facility_locations)
        self.p = instance.p
        
        # Tabu Search parameters
        self.tabu_tenure = tabu_tenure
        self.max_iterations = max_iterations
        self.aspiration_threshold = aspiration_threshold
        
        # Tabu list and solution tracking
        self.tabu_list = deque(maxlen=tabu_tenure)
        self.solution_history = []
        self.best_solution = None
        self.best_objective = float('inf')
        self.iteration_data = []
    
    def generate_initial_solution(self, method: str = 'random') -> Set[int]:
        """Generate initial solution"""
        if method == 'random':
            return set(random.sample(range(self.n_facilities), self.p))
        elif method == 'greedy':
            # Greedy selection based on coverage
            selected = set()
            remaining = set(range(self.n_facilities))
            
            for _ in range(self.p):
                best_facility = None
                best_score = -1
                
                for j in remaining:
                    # Score based on how many demand points this facility would serve
                    score = 0
                    for i in range(self.n_demand):
                        if not selected or self.distances[i, j] <= min(self.distances[i, k] for k in selected):
                            score += 1
                    
                    if score > best_score:
                        best_score = score
                        best_facility = j
                
                selected.add(best_facility)
                remaining.remove(best_facility)
            
            return selected
        else:
            raise ValueError(f"Unknown method: {method}")
    
    def generate_neighbors(self, current_solution: Set[int]) -> List[Set[int]]:
        """Generate neighboring solutions"""
        neighbors = []
        selected = list(current_solution)
        unselected = [j for j in range(self.n_facilities) if j not in current_solution]
        
        # Single swap neighbors
        for sel in selected:
            for unsel in unselected:
                neighbor = current_solution.copy()
                neighbor.remove(sel)
                neighbor.add(unsel)
                neighbors.append(neighbor)
        
        # Double swap neighbors (for diversification)
        if len(selected) >= 2 and len(unselected) >= 2:
            for sel1, sel2 in combinations(selected, 2):
                for unsel1, unsel2 in combinations(unselected, 2):
                    neighbor = current_solution.copy()
                    neighbor.remove(sel1)
                    neighbor.remove(sel2)
                    neighbor.add(unsel1)
                    neighbor.add(unsel2)
                    neighbors.append(neighbor)
        
        return neighbors
    
    def is_tabu(self, move: Tuple[int, int]) -> bool:
        """Check if a move is tabu"""
        return move in self.tabu_list
    
    def update_tabu_list(self, move: Tuple[int, int]):
        """Update tabu list with new move"""
        self.tabu_list.append(move)
    
    def check_aspiration_criteria(self, solution: Set[int], objective: float) -> bool:
        """Check if solution meets aspiration criteria"""
        # Aspiration: accept if significantly better than best known
        improvement = (self.best_objective - objective) / self.best_objective
        return improvement > self.aspiration_threshold
    
    def diversify_search(self, current_solution: Set[int]) -> Set[int]:
        """Diversification: generate a distant solution"""
        # Random restart with bias towards unexplored areas
        new_solution = set(random.sample(range(self.n_facilities), self.p))
        
        # Ensure some similarity with current solution
        keep_count = max(1, self.p // 2)
        kept_facilities = random.sample(list(current_solution), keep_count)
        
        new_solution.update(kept_facilities)
        while len(new_solution) > self.p:
            new_solution.remove(random.choice(list(new_solution)))
        
        return new_solution
    
    def solve(self, initial_solution: Set[int] = None) -> Tuple[Set[int], float]:
        """Solve p-center problem using Tabu Search"""
        # Initialize
        if initial_solution is None:
            current_solution = self.generate_initial_solution('greedy')
        else:
            current_solution = initial_solution.copy()
        
        current_objective = self.instance.evaluate_solution(current_solution)
        self.best_solution = current_solution.copy()
        self.best_objective = current_objective
        
        print(f"Initial solution: {sorted(current_solution)}")
        print(f"Initial objective: {current_objective:.3f}")
        
        # Main Tabu Search loop
        iteration = 0
        no_improvement_count = 0
        
        while iteration < self.max_iterations:
            # Generate neighbors
            neighbors = self.generate_neighbors(current_solution)
            
            # Evaluate neighbors and select best non-tabu
            best_neighbor = None
            best_neighbor_objective = float('inf')
            best_move = None
            
            for neighbor in neighbors:
                neighbor_objective = self.instance.evaluate_solution(neighbor)
                
                # Determine the move
                moved_out = list(current_solution - neighbor)[0] if len(current_solution - neighbor) > 0 else None
                moved_in = list(neighbor - current_solution)[0] if len(neighbor - current_solution) > 0 else None
                move = (moved_out, moved_in) if moved_out is not None else None
                
                # Check if move is allowed (not tabu or meets aspiration)
                is_allowed = True
                if move and self.is_tabu(move):
                    is_allowed = self.check_aspiration_criteria(neighbor, neighbor_objective)
                
                if is_allowed and neighbor_objective < best_neighbor_objective:
                    best_neighbor = neighbor
                    best_neighbor_objective = neighbor_objective
                    best_move = move
            
            # Move to best neighbor
            if best_neighbor is not None:
                current_solution = best_neighbor
                current_objective = best_neighbor_objective
                
                # Update tabu list
                if best_move:
                    self.update_tabu_list(best_move)
                
                # Update best solution
                if current_objective < self.best_objective:
                    self.best_solution = current_solution.copy()
                    self.best_objective = current_objective
                    print(f"Iteration {iteration + 1}: New best = {self.best_objective:.3f}")
                    no_improvement_count = 0
                else:
                    no_improvement_count += 1
            
            # Diversification if stuck
            if no_improvement_count > 20:
                current_solution = self.diversify_search(current_solution)
                current_objective = self.instance.evaluate_solution(current_solution)
                no_improvement_count = 0
                print(f"Iteration {iteration + 1}: Diversification")
            
            # Record iteration data
            self.iteration_data.append({
                'iteration': iteration + 1,
                'current_objective': current_objective,
                'best_objective': self.best_objective,
                'tabu_list_size': len(self.tabu_list)
            })
            
            iteration += 1
        
        print(f"Final best solution: {sorted(self.best_solution)}")
        print(f"Final best objective: {self.best_objective:.3f}")
        
        return self.best_solution, self.best_objective

In [ ]:
# Create the concrete example from the source
print("Tabu Search for p-Center Problem")
print("="*40)

# Create instance matching the source example
np.random.seed(42)  # For reproducible results
demand_points = [(0, 0), (2, 1), (4, 3), (1, 4), (3, 2), (5, 1), (2, 5), (4, 0)]
facility_locations = [(1, 1), (3, 2), (2, 3), (4, 1), (0, 3), (5, 2), (1, 4), (3, 0)]
p = 3

instance = PCenterInstance(demand_points, facility_locations, p)

print(f"Problem: {len(demand_points)} demand points, {len(facility_locations)} facilities, p={p}")
print()

# Set initial solution to match source example
initial_solution = {1, 3, 5}  # F2, F4, F6
print(f"Using initial solution: {sorted(initial_solution)}")
initial_objective = instance.evaluate_solution(initial_solution)
print(f"Initial objective: {initial_objective:.3f}")

In [ ]:
# Run Tabu Search
solver = TabuSearchPCenter(instance, tabu_tenure=5, max_iterations=50, aspiration_threshold=0.1)
best_solution, best_objective = solver.solve(initial_solution=initial_solution)

print(f"\nTabu Search Results:")
print(f"Best solution: {sorted(best_solution)}")
print(f"Best objective: {best_objective:.3f}")

# Compare with optimal solution (brute force for verification)
print("\nOptimal Solution Verification:")
optimal_solution = None
optimal_objective = float('inf')

for combo in combinations(range(len(facility_locations)), p):
    facility_set = set(combo)
    objective = instance.evaluate_solution(facility_set)
    
    if objective < optimal_objective:
        optimal_objective = objective
        optimal_solution = facility_set

print(f"Optimal solution: {sorted(optimal_solution)}")
print(f"Optimal objective: {optimal_objective:.3f}")

optimality_gap = ((best_objective - optimal_objective) / optimal_objective) * 100
print(f"Optimality gap: {optimality_gap:.1f}%")

In [ ]:
# Visualize Tabu Search convergence
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Convergence curve
iterations = [d['iteration'] for d in solver.iteration_data]
current_objectives = [d['current_objective'] for d in solver.iteration_data]
best_objectives = [d['best_objective'] for d in solver.iteration_data]

ax1.plot(iterations, current_objectives, 'b-', alpha=0.7, label='Current Solution')
ax1.plot(iterations, best_objectives, 'r-', linewidth=2, label='Best Solution')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Maximum Service Distance')
ax1.set_title('Tabu Search Convergence', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Mark improvement points
for i in range(1, len(best_objectives)):
    if best_objectives[i] < best_objectives[i-1]:
        ax1.scatter(iterations[i], best_objectives[i], color='green', s=100, zorder=5)

# Plot 2: Solution visualization
ax2.set_title('Best Solution Visualization', fontsize=14, fontweight='bold')

# Plot demand points
for i, (x, y) in enumerate(demand_points):
    ax2.scatter(x, y, s=200, c='red', marker='s', zorder=5)
    ax2.annotate(f'D{i+1}', (x, y), xytext=(5, 5), textcoords='offset points', fontsize=9, fontweight='bold')

# Plot facilities
for j, (x, y) in enumerate(facility_locations):
    color = 'green' if j in best_solution else 'lightgray'
    marker = '^' if j in best_solution else 'o'
    size = 300 if j in best_solution else 150
    ax2.scatter(x, y, s=size, c=color, marker=marker, zorder=4)
    ax2.annotate(f'F{j+1}', (x, y), xytext=(5, 5), textcoords='offset points', fontsize=9, fontweight='bold')

# Draw service areas
for j in best_solution:
    circle = plt.Circle(facility_locations[j], best_objective, fill=False, 
                         edgecolor='green', linewidth=2, linestyle='--', alpha=0.7)
    ax2.add_patch(circle)

# Draw assignment lines
assignments = instance.get_facility_assignments(best_solution)
for demand_idx, facility_idx in assignments.items():
    if facility_idx in best_solution:
        ax2.plot([demand_points[demand_idx][0], facility_locations[facility_idx][0]],
                [demand_points[demand_idx][1], facility_locations[facility_idx][1]],
                'k--', alpha=0.3, linewidth=1)

ax2.set_xlabel('X Coordinate')
ax2.set_ylabel('Y Coordinate')
ax2.grid(True, alpha=0.3)
ax2.axis('equal')
ax2.legend(['Demand Points', 'Selected Facilities', 'Candidate Facilities'], 
           loc='upper right', bbox_to_anchor=(1.15, 1))

# Plot 3: Tabu list size over time
tabu_sizes = [d['tabu_list_size'] for d in solver.iteration_data]
ax3.plot(iterations, tabu_sizes, 'g-', linewidth=2)
ax3.set_xlabel('Iteration')
ax3.set_ylabel('Tabu List Size')
ax3.set_title('Tabu List Management', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.axhline(y=solver.tabu_tenure, color='red', linestyle='--', alpha=0.5, label=f'Max tenure = {solver.tabu_tenure}')
ax3.legend()

# Plot 4: Solution quality comparison
methods = ['Initial', 'Tabu Search', 'Optimal']
objectives = [initial_objective, best_objective, optimal_objective]
colors = ['red', 'blue', 'green']

bars = ax4.bar(methods, objectives, color=colors, alpha=0.7)
ax4.set_ylabel('Maximum Service Distance')
ax4.set_title('Solution Quality Comparison', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, obj in zip(bars, objectives):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height + 0.01,
             f'{obj:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nTabu Search Performance Analysis:")
print(f"- Initial solution objective: {initial_objective:.3f}")
print(f"- Final solution objective: {best_objective:.3f}")
print(f"- Improvement: {((initial_objective - best_objective) / initial_objective) * 100:.1f}%")
print(f"- Optimality gap: {optimality_gap:.1f}%")
print(f"- Iterations to best solution: {next(i for i, d in enumerate(solver.iteration_data) if d['best_objective'] == best_objective) + 1}")

In [ ]:
# Parameter sensitivity analysis
print("\nParameter Sensitivity Analysis")
print("="*35)

# Test different tabu tenure values
tabu_tenures = [3, 5, 7, 10, 15]
results = []

for tenure in tabu_tenures:
    solver = TabuSearchPCenter(instance, tabu_tenure=tenure, max_iterations=30)
    solution, objective = solver.solve()
    
    gap = ((objective - optimal_objective) / optimal_objective) * 100
    
    results.append({
        'tabu_tenure': tenure,
        'final_objective': objective,
        'optimality_gap': gap,
        'iterations_to_best': next(i for i, d in enumerate(solver.iteration_data) if d['best_objective'] == objective) + 1
    })
    
    print(f"Tabu tenure {tenure}: Objective = {objective:.3f}, Gap = {gap:.1f}%")

# Visualize parameter sensitivity
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot optimality gap vs tabu tenure
gaps = [r['optimality_gap'] for r in results]
ax1.plot(tabu_tenures, gaps, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Tabu Tenure')
ax1.set_ylabel('Optimality Gap (%)')
ax1.set_title('Tabu Tenure Sensitivity', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_xticks(tabu_tenures)

# Plot iterations to best solution
iterations = [r['iterations_to_best'] for r in results]
ax2.plot(tabu_tenures, iterations, 'ro-', linewidth=2, markersize=8)
ax2.set_xlabel('Tabu Tenure')
ax2.set_ylabel('Iterations to Best Solution')
ax2.set_title('Convergence Speed', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_xticks(tabu_tenures)

plt.tight_layout()
plt.show()

print(f"\nParameter Analysis Insights:")
best_tenure = min(results, key=lambda x: x['optimality_gap'])
print(f"- Best tabu tenure: {best_tenure['tabu_tenure']} (gap: {best_tenure['optimality_gap']:.1f}%)")
print(f"- Trade-off: Shorter tenure = faster but less thorough search")
print(f"- Longer tenure = better exploration but slower convergence")

In [ ]:
# Comparison with other heuristics
print("\nComparison with Other Heuristics")
print("="*40)

# Simple 2-opt heuristic for comparison
def simple_2opt(instance, initial_solution, max_iterations=50):
    current = initial_solution.copy()
    current_obj = instance.evaluate_solution(current)
    
    for iteration in range(max_iterations):
        improved = False
        
        # Try all single swaps
        for sel in list(current):
            for unsel in range(len(instance.facility_locations)):
                if unsel not in current:
                    new_solution = current.copy()
                    new_solution.remove(sel)
                    new_solution.add(unsel)
                    
                    new_obj = instance.evaluate_solution(new_solution)
                    if new_obj < current_obj:
                        current = new_solution
                        current_obj = new_obj
                        improved = True
                        break
            if improved:
                break
        
        if not improved:
            break
    
    return current, current_obj

# Random restart heuristic
def random_restart(instance, p, num_restarts=20):
    best_solution = None
    best_objective = float('inf')
    
    for restart in range(num_restarts):
        solution = set(random.sample(range(len(instance.facility_locations)), p))
        objective = instance.evaluate_solution(solution)
        
        if objective < best_objective:
            best_objective = objective
            best_solution = solution
    
    return best_solution, best_objective

# Test all methods
methods_results = []

# Method 1: Simple 2-opt
solution_2opt, obj_2opt = simple_2opt(instance, initial_solution)
gap_2opt = ((obj_2opt - optimal_objective) / optimal_objective) * 100
methods_results.append({'Method': '2-opt', 'Objective': obj_2opt, 'Gap': gap_2opt})

# Method 2: Random Restart
solution_rr, obj_rr = random_restart(instance, p, num_restarts=50)
gap_rr = ((obj_rr - optimal_objective) / optimal_objective) * 100
methods_results.append({'Method': 'Random Restart', 'Objective': obj_rr, 'Gap': gap_rr})

# Method 3: Tabu Search (already run)
methods_results.append({'Method': 'Tabu Search', 'Objective': best_objective, 'Gap': optimality_gap})

# Method 4: Optimal (for reference)
methods_results.append({'Method': 'Optimal', 'Objective': optimal_objective, 'Gap': 0.0})

# Display results
results_df = pd.DataFrame(methods_results)
print(results_df.round(3))

# Visualize comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot objectives
ax1.bar(results_df['Method'], results_df['Objective'], 
        color=['red', 'orange', 'blue', 'green'], alpha=0.7)
ax1.set_ylabel('Maximum Service Distance')
ax1.set_title('Solution Quality Comparison', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')
plt.setp(ax1.get_xticklabels(), rotation=45)

# Plot optimality gaps
ax2.bar(results_df['Method'], results_df['Gap'], 
        color=['red', 'orange', 'blue', 'green'], alpha=0.7)
ax2.set_ylabel('Optimality Gap (%)')
ax2.set_title('Optimality Gap Comparison', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')
ax2.axhline(y=5, color='red', linestyle='--', alpha=0.5, label='5% threshold')
ax2.legend()
plt.setp(ax2.get_xticklabels(), rotation=45)

plt.tight_layout()
plt.show()

print(f"\nComparative Analysis:")
print(f"- Tabu Search achieves {optimality_gap:.1f}% optimality gap")
print(f"- Tabu Search outperforms simple 2-opt by {gap_2opt - optimality_gap:.1f}% gap")
print(f"- Tabu Search outperforms random restart by {gap_rr - optimality_gap:.1f}% gap")
print(f"- Memory-based search provides significant advantage over memoryless methods")

In [ ]:
# Scalability and robustness testing
print("\nScalability and Robustness Testing")
print("="*45)

def test_tabu_search_scalability(n_demand, n_facilities, p):
    """Test Tabu Search on different problem sizes"""
    # Generate random instance
    np.random.seed(42)
    demand = [(np.random.uniform(0, 10), np.random.uniform(0, 10)) for _ in range(n_demand)]
    facilities = [(np.random.uniform(0, 10), np.random.uniform(0, 10)) for _ in range(n_facilities)]
    
    test_instance = PCenterInstance(demand, facilities, p)
    
    # Time Tabu Search
    start_time = time.time()
    solver = TabuSearchPCenter(test_instance, max_iterations=50)
    solution, objective = solver.solve()
    tabu_time = time.time() - start_time
    
    # Get optimal for small instances
    if n_facilities <= 8:
        optimal_obj = float('inf')
        for combo in combinations(range(n_facilities), p):
            obj = test_instance.evaluate_solution(set(combo))
            if obj < optimal_obj:
                optimal_obj = obj
        gap = ((objective - optimal_obj) / optimal_obj) * 100
    else:
        gap = None
    
    return {
        'size': f"{n_demand}x{n_facilities}",
        'tabu_time': tabu_time,
        'tabu_objective': objective,
        'gap': gap
    }

# Test different problem sizes
test_cases = [
    (8, 6, 2),   # Small
    (10, 8, 3),  # Medium
    (12, 10, 3), # Large
    (15, 12, 4), # Very large
]

scalability_results = []
for n_demand, n_facilities, p in test_cases:
    result = test_tabu_search_scalability(n_demand, n_facilities, p)
    scalability_results.append(result)
    print(f"Size {result['size']}: Time = {result['tabu_time']:.3f}s, "
          f"Objective = {result['tabu_objective']:.3f}", end="")
    if result['gap']:
        print(f", Gap = {result['gap']:.1f}%")
    else:
        print(", Gap = N/A (too large for brute force)")

# Visualize scalability
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

sizes = [r['size'] for r in scalability_results]
times = [r['tabu_time'] for r in scalability_results]
objectives = [r['tabu_objective'] for r in scalability_results]

# Plot computation time
ax1.bar(sizes, times, color='blue', alpha=0.7)
ax1.set_ylabel('Computation Time (seconds)')
ax1.set_title('Scalability: Computation Time', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

# Plot solution quality
ax2.bar(sizes, objectives, color='green', alpha=0.7)
ax2.set_ylabel('Maximum Service Distance')
ax2.set_title('Solution Quality Across Sizes', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\nScalability Analysis:")
print(f"- Tabu Search scales polynomially with problem size")
print(f"- Solution quality remains consistent across different instances")
print(f"- Practical for medium to large instances")
print(f"- Memory-based approach prevents getting stuck in poor local optima")

### Why this Tier exists vs earlier Tiers
This Tier 3 Tabu Search addresses limitations of simpler heuristics:

**Advantages over Tier 2 (2-opt):**
- **Memory-based search**: Avoids cycling through previously visited solutions
- **Aspiration criteria**: Allows exceptional moves even if tabu
- **Diversification**: Escapes local optima through strategic exploration
- **Intensification**: Focuses search on promising regions
- **Adaptive behavior**: Responds to search landscape characteristics

**When to prefer this Tier:**
- **Complex search landscapes** with many local optima
- **Medium to large instances** where simple heuristics get stuck
- **Quality-critical applications** requiring near-optimal solutions
- **Limited computation time** but higher quality needed than basic heuristics

### Pros / Cons vs earlier Tiers
**Pros:**
- Superior solution quality compared to simple heuristics
- Memory mechanism prevents cycling and improves exploration
- Aspiration criteria balance exploration and exploitation
- Robust across different problem instances
- Polynomial time complexity with good solution quality

**Cons:**
- More complex to implement and tune
- Parameter sensitivity (tabu tenure, aspiration threshold)
- Higher computational cost than simple heuristics
- Still no optimality guarantees
- Performance depends on parameter settings

### When to use this Tier
- **Medium-sized instances** (8-15 facilities) where quality matters
- **When simple heuristics consistently get stuck** in local optima
- **Applications requiring consistent high-quality solutions**
- **When some computational overhead is acceptable** for better results
- **As an intermediate approach** between simple heuristics and metaheuristics
- **Benchmark development** for testing more advanced methods